In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from pydantic import BaseModel,Field
from langchain_groq import  ChatGroq
from langgraph.graph import StateGraph, START, END
from typing import Literal

#libs for agent
from langchain_community.utilities import GoogleSearchAPIWrapper
from langchain.agents import create_agent
from langchain.tools import tool

In [ ]:
## coding , google search , wheather

class FlowState(BaseModel):
    question:str=Field(...,description="ask question")
    category:Literal['coding','google_search','weather']=Field(default="google_search")
    answer:str=Field(default="")

In [ ]:
class QuestionCategory(BaseModel):
    category:Literal['coding','google_search','weather']=Field(default="google_search", description="question category")

In [ ]:
llm= ChatGroq(model="llma-3.5-mini", )

In [ ]:
# define your agent googlesearch ,wheather
search=GoogleSearchAPIWrapperserper_api_key="sdfvbvfc"()
tools=[search.run]

google_agent=create_agent(
    model=llm,
    tools=tools,
    system_prompt="you are a agent and can search for any question on google."
)

# wheather agent
@tool
def get_weather(city:str):
    """it provide real time weather details info
 
    """
    return f"current temperature in {city} is 30 degree celsius"

wheather_agent=create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="you are a agent and can give weather information of any city realtime"
)


In [ ]:
def check_question_category(state:FlowState)->FlowState:
    st_llm=llm.with_structured_output(QuestionCategory)
    res=st_llm.invoke(f"i want to know the category of my questions is: {state.question}. if you are not sure just give google_serach as category")
    state.category=res.category
    return state

In [ ]:
# flow=FlowState(question="what is the weather today in hyderabad?")
# check_question_category(flow)

In [ ]:
def route(state:FlowState)-> Literal['coding','google_search','weather']:
    return state.category

In [ ]:
def coding_node(state:FlowState)->FlowState:
    print("coding node")
    res=llm.invoke(f"you are a coding expert: {state.question}")
    state.answer=res.content
    return state


def wheather_node(state:FlowState)->FlowState:
    res=wheather_agent.invoke({"messages":[
       {"role:user","content:state.question"}
    ]})
    state.answer=res["messages"][-1].content
 
    return state


def google_search_node(state:FlowState)->FlowState:
    res=google_agent.invoke({"messages":[
       {"role:user","content:state.question"}
    ]})
    state.answer=res["messages"][-1].content
    return state

In [ ]:
graph=StateGraph(FlowState)
graph.add_node("check_question_category", check_question_category)
graph.add_node("coding",coding_node)
graph.add_node("weather",wheather_node)
graph.add_node("google_search",google_search_node)

graph.add_edge(START,"check_question_category")
graph.add_conditional_edges("check_question_category",route)
graph.add_edge("coding",END)
graph.add_edge("weather",END)
graph.add_edge("google_search",END)
graph=graph.compile()

In [ ]:
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
# response=graph.invoke({"question":"what is the weather today in hyderabad?"})
response=graph.invoke({"question":"what is 2+2?"})